# Phase 4: Train once, save everything (run this FIRST)

**This is the only notebook that trains models.** Notebooks 01-03 load the seed-42 run from `results/`. Trains the CNN baseline and the transformer under N training seeds on the same fixed subject split, and saves every metric the write-up needs to `results/*.json` (macro/per-class F1, accuracy, kappa, normalised confusion matrix, transition vs. stable F1, per-subject F1, calibration stats). Each (model, seed) is saved the moment it finishes and skipped on re-run, so if Colab disconnects, just re-run the notebook.

The seed-to-seed spread tells us whether the CNN-vs-transformer gap in `02_transformer.ipynb` is real or training noise.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

import numpy as np
from scipy import stats
from torch.utils.data import DataLoader

from src.config import BATCH_SIZE, CASSETTE_DIR, LABEL_NAMES, get_device
from src.data import SleepEDFDataset, build_epoch_arrays, discover_subjects, get_subject_split
from src.experiments import RESULTS_DIR, load_all, run_seeds, summarize

SEEDS = [42, 43, 44]
device = get_device()
print(f"device: {device}")

## 0. Check data and keep results on Drive

Uses the Sleep-EDF files already linked from Drive (nothing is downloaded). Expect ~153 PSG + 153 hypnogram files. `results/` is linked to Drive so finished runs persist and a re-run resumes where it stopped.

In [ ]:
# Colab: expects the usual setup already done (Drive mounted, repo cloned, `data` -> MyDrive/eeg-sleep-data).
# This cell (1) checks the data is visible and (2) keeps results/ on Drive so runs survive a disconnect.
import os

from src.config import CASSETTE_DIR, ROOT_DIR

n_psg = len(list(CASSETTE_DIR.glob("SC*-PSG.edf")))
n_hyp = len(list(CASSETTE_DIR.glob("SC*-Hypnogram.edf")))
print(f"data dir: {CASSETTE_DIR}  ->  {n_psg} PSG / {n_hyp} hypnogram files")
assert n_psg > 0, "no data found -- mount Drive and link data -> MyDrive/eeg-sleep-data first"

DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/eeg-sleep-results")
if DRIVE_RESULTS_DIR.parent.exists() and not (ROOT_DIR / "results").exists():
    DRIVE_RESULTS_DIR.mkdir(exist_ok=True)
    os.symlink(DRIVE_RESULTS_DIR, ROOT_DIR / "results")
print("results dir:", (ROOT_DIR / "results").resolve())

## 1. Build the split (identical to notebooks 01-03)

In [ ]:
subjects = discover_subjects(CASSETTE_DIR)
subject_ids = [s.subject_id for s in subjects]
train_ids, val_ids, test_ids = get_subject_split(subject_ids)
print(f"subjects: {len(subject_ids)}  (train {len(train_ids)} / val {len(val_ids)} / test {len(test_ids)})")

train_epochs, train_labels, _ = build_epoch_arrays(subjects, train_ids)
val_epochs, val_labels, _ = build_epoch_arrays(subjects, val_ids)
test_epochs, test_labels, test_subject_ids = build_epoch_arrays(subjects, test_ids)

train_loader = DataLoader(SleepEDFDataset(train_epochs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepEDFDataset(val_epochs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SleepEDFDataset(test_epochs, test_labels), batch_size=BATCH_SIZE, shuffle=False)
print(f"train/val/test epochs: {len(train_labels)} / {len(val_labels)} / {len(test_labels)}")

## 2. Train every (model, seed)\n\nResumable: finished runs are skipped.

In [ ]:
run_seeds(train_loader, val_loader, test_loader, test_subject_ids, seeds=SEEDS, device=device)

## 3. Aggregate: mean ± std across seeds

In [ ]:
grouped = load_all()
table = summarize(grouped)
print(table)

## 4. Is the transformer-vs-CNN gap real?\n\nPaired across seeds (same seed index for both models). With only a handful of seeds this has little power -- treat a large p-value as "can't distinguish", not "proven equal".

In [ ]:
by_seed = {name: {r["seed"]: r["macro_f1"] for r in runs} for name, runs in grouped.items()}
common = sorted(set(by_seed["cnn"]) & set(by_seed["transformer"]))
cnn = np.array([by_seed["cnn"][s] for s in common])
tfm = np.array([by_seed["transformer"][s] for s in common])
diff = tfm - cnn

print(f"seeds: {common}")
print(f"CNN         macro F1: {cnn.mean():.4f} +/- {cnn.std(ddof=1):.4f}")
print(f"Transformer macro F1: {tfm.mean():.4f} +/- {tfm.std(ddof=1):.4f}")
print(f"mean diff (transformer - CNN): {diff.mean():+.4f}  (std {diff.std(ddof=1):.4f})")
t, p = stats.ttest_rel(tfm, cnn)
print(f"paired t-test: t={t:.3f}, p={p:.3f}")

print("\nper-class F1 diff (transformer - CNN), mean over seeds:")
for c in LABEL_NAMES:
    d = np.mean([[r["per_class_f1"][c] for r in grouped["transformer"] if r["seed"] == s][0]
                 - [r["per_class_f1"][c] for r in grouped["cnn"] if r["seed"] == s][0] for s in common])
    print(f"  {c:<4}{d:+.4f}")

## 5. Other numbers for `results.md`\n\nTransformer seed-averaged confusion matrix, per-subject spread, and calibration gap.

In [ ]:
tf_runs = grouped["transformer"]
cm_mean = np.mean([r["confusion_normalized"] for r in tf_runs], axis=0)
print("transformer confusion matrix (row-normalised, mean over seeds):")
print("      " + "".join(f"{n:>7}" for n in LABEL_NAMES))
for name, row in zip(LABEL_NAMES, cm_mean):
    print(f"{name:<6}" + "".join(f"{v:>7.2f}" for v in row))

print("\ntop confusion pairs by seed (true, pred, count):")
for r in tf_runs:
    print(f"  seed {r['seed']}: {r['top_confusion_pairs'][:3]}")

subj = np.array([[r["per_subject_f1"][k] for k in sorted(r["per_subject_f1"])] for r in tf_runs]).mean(axis=0)
print(f"\nper-subject macro F1 (seed-mean): min {subj.min():.3f}, median {np.median(subj):.3f}, max {subj.max():.3f}")

print(f"mean confidence, correct:   {np.mean([r['mean_conf_correct'] for r in tf_runs]):.3f}")
print(f"mean confidence, incorrect: {np.mean([r['mean_conf_incorrect'] for r in tf_runs]):.3f}")

## 6. Save everything

On Colab the filesystem is wiped when the runtime recycles. Copy `results/` to Drive (or download it), then commit the JSON files:

```python
!cp -r ../results /content/drive/MyDrive/eeg_results   # if Drive is mounted
```